# Backtest: Using Backpack Exchange Connector

Tutorial for [NautilusTrader](https://nautilustrader.io/docs/) demonstrating how to use the official Backpack Exchange connector for backtesting.

[View source on GitHub](https://github.com/nautechsystems/nautilus_trader/blob/develop/docs/tutorials/backtest_backpack_connector.ipynb).

## Overview

This tutorial demonstrates how to use the **official Backpack Exchange connector** to:
1. Set up the Backpack HTTP client and fetch historical data
2. Create proper instrument definitions using the connector
3. Process and store data in a catalog
4. Run a backtest with the fetched data
5. Analyze results using NautilusTrader's reporting tools

This approach uses the production-ready connector infrastructure, which provides better error handling and integration with the NautilusTrader ecosystem.

## Prerequisites

- Python 3.11+ installed
- [JupyterLab](https://jupyter.org/) or similar installed (`pip install -U jupyterlab`)
- [NautilusTrader](https://pypi.org/project/nautilus_trader/) latest release installed (`pip install -U nautilus_trader`)
- Optional: Backpack Exchange API credentials (not required for public data)

## Imports

Import all necessary components from NautilusTrader and the Backpack adapter:

In [ ]:
import asyncio
import base64
import os
import shutil
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import pandas as pd

# Backpack specific imports
from nautilus_trader.adapters.backpack.common.constants import BACKPACK_VENUE
from nautilus_trader.adapters.backpack.http.client import BackpackHttpClient
from nautilus_trader.adapters.backpack.http.history import BackpackHistoryHttpAPI
from nautilus_trader.adapters.backpack.providers import BackpackInstrumentProvider

# NautilusTrader core imports
from nautilus_trader.backtest.node import BacktestDataConfig
from nautilus_trader.backtest.node import BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.backtest.node import BacktestRunConfig
from nautilus_trader.backtest.node import BacktestVenueConfig
from nautilus_trader.common.component import LiveClock
from nautilus_trader.config import ImportableStrategyConfig
from nautilus_trader.config import InstrumentProviderConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model.currencies import Currency
from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import BarType
from nautilus_trader.model.data import BarSpecification
from nautilus_trader.model.enums import BarAggregation
from nautilus_trader.model.enums import PriceType
from nautilus_trader.model.enums import AggregationSource
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.identifiers import Symbol
from nautilus_trader.model.instruments import CurrencyPair
from nautilus_trader.model.objects import Money
from nautilus_trader.model.objects import Price
from nautilus_trader.model.objects import Quantity
from nautilus_trader.persistence.catalog import ParquetDataCatalog

## Initialize Backpack HTTP Client

The `BackpackHttpClient` handles all HTTP communication with the Backpack API. For public endpoints, we can use dummy credentials:

In [ ]:
# Initialize clock
clock = LiveClock()

# Create a dummy base64 encoded secret (32 bytes) for public endpoints
# For actual trading, use real credentials from environment variables
dummy_secret = base64.b64encode(b"0" * 32).decode('utf-8')

# Create HTTP client
http_client = BackpackHttpClient(
    clock=clock,
    api_key="dummy_api_key",  # Dummy key for public endpoints
    api_secret=dummy_secret,    # Dummy secret for public endpoints
    base_url="https://api.backpack.exchange",
)

print("HTTP client initialized")

## Using BackpackHistoryHttpAPI

The `BackpackHistoryHttpAPI` provides methods to fetch historical data from Backpack:

In [ ]:
# Create history API instance
history_api = BackpackHistoryHttpAPI(http_client)

# Define time range (last 30 days of hourly data)
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=30)

# Convert to seconds (Backpack API uses seconds, not milliseconds)
start_ts = int(start_time.timestamp())
end_ts = int(end_time.timestamp())

print(f"Fetching data from {start_time} to {end_time}")

# Fetch klines using the connector
klines = await history_api.fetch_klines_history(
    symbol="BTC_USDC",
    interval="1h",
    start_time=start_ts,
    end_time=end_ts,
    limit=1000,
)

print(f"Fetched {len(klines)} klines from Backpack")
if klines:
    print(f"Sample kline: {klines[0]}")

## Create Instrument Definition

We'll create a proper instrument definition using the actual data precision:

In [ ]:
def create_backpack_instrument(
    symbol: str = "BTC_USDC",
    klines: list = None,
) -> CurrencyPair:
    """Create a Backpack instrument with auto-detected precision."""
    
    base, quote = symbol.split("_")
    
    # Detect precision from actual data
    price_precision = 1
    size_precision = 3
    
    if klines and len(klines) > 0:
        first_kline = klines[0]
        
        # Get sample values
        if isinstance(first_kline, dict):
            sample_price = str(first_kline.get('open', 0))
            sample_volume = str(first_kline.get('volume', 0))
        else:
            # List format from API
            sample_price = str(first_kline[1]) if len(first_kline) > 1 else "0"
            sample_volume = str(first_kline[5]) if len(first_kline) > 5 else "0"
        
        # Detect price precision
        if '.' in sample_price:
            decimal_places = len(sample_price.split('.')[1].rstrip('0'))
            price_precision = max(1, decimal_places)
        
        # Detect volume precision
        if '.' in sample_volume:
            decimal_places = len(sample_volume.split('.')[1].rstrip('0'))
            size_precision = max(1, decimal_places)
    
    print(f"Detected price precision: {price_precision}, size precision: {size_precision}")
    
    # Create price and size increments
    price_increment_str = "0." + "0" * (price_precision - 1) + "1" if price_precision > 0 else "1"
    size_increment_str = "0." + "0" * (size_precision - 1) + "1" if size_precision > 0 else "1"
    
    return CurrencyPair(
        instrument_id=InstrumentId(
            symbol=Symbol(symbol),
            venue=BACKPACK_VENUE,
        ),
        raw_symbol=Symbol(symbol),
        base_currency=Currency.from_str(base),
        quote_currency=Currency.from_str(quote),
        price_precision=price_precision,
        size_precision=size_precision,
        price_increment=Price.from_str(price_increment_str),
        size_increment=Quantity.from_str(size_increment_str),
        lot_size=None,
        max_quantity=Quantity.from_str("10000"),
        min_quantity=Quantity.from_str("0.0001"),
        max_notional=None,
        min_notional=Money(10, Currency.from_str(quote)),
        max_price=Price.from_str("10000000"),
        min_price=Price.from_str("0.01"),
        margin_init=Decimal("0"),
        margin_maint=Decimal("0"),
        maker_fee=Decimal("0.0002"),  # 0.02%
        taker_fee=Decimal("0.0005"),  # 0.05%
        ts_event=0,
        ts_init=0,
    )

# Create instrument using fetched data
instrument = create_backpack_instrument("BTC_USDC", klines)
print(f"Created instrument: {instrument.id}")

## Process Klines to Bars

Convert the raw klines data into NautilusTrader Bar objects:

In [ ]:
def process_klines_to_bars(
    klines: list,
    instrument: CurrencyPair,
    bar_type: BarType,
) -> list[Bar]:
    """Process raw klines data into NautilusTrader Bar objects."""
    
    bars = []
    
    for kline in klines:
        # Parse kline data based on format
        if isinstance(kline, dict):
            # Dictionary format from API
            timestamp_str = kline.get('start', kline.get('timestamp', 0))
            open_price = kline.get('open', 0)
            high_price = kline.get('high', 0)
            low_price = kline.get('low', 0)
            close_price = kline.get('close', 0)
            volume = kline.get('volume', 0)
        else:
            # List format: [timestamp, open, high, low, close, volume, ...]
            timestamp_str = kline[0] if len(kline) > 0 else 0
            open_price = kline[1] if len(kline) > 1 else 0
            high_price = kline[2] if len(kline) > 2 else 0
            low_price = kline[3] if len(kline) > 3 else 0
            close_price = kline[4] if len(kline) > 4 else 0
            volume = kline[5] if len(kline) > 5 else 0
        
        # Parse timestamp
        if isinstance(timestamp_str, str):
            # Parse string timestamp
            from dateutil import parser
            dt = parser.parse(timestamp_str)
            timestamp_ms = int(dt.timestamp() * 1000)
        else:
            # Numeric timestamp (already in ms or seconds)
            timestamp_ms = int(timestamp_str)
            # If it looks like seconds, convert to ms
            if timestamp_ms < 10**10:
                timestamp_ms = timestamp_ms * 1000
        
        # Convert to nanoseconds
        ts_event = timestamp_ms * 1_000_000
        ts_init = ts_event
        
        bar = Bar(
            bar_type=bar_type,
            open=Price.from_str(str(open_price)),
            high=Price.from_str(str(high_price)),
            low=Price.from_str(str(low_price)),
            close=Price.from_str(str(close_price)),
            volume=Quantity.from_str(str(volume)),
            ts_event=ts_event,
            ts_init=ts_init,
        )
        bars.append(bar)
    
    return bars

# Define bar type
bar_type = BarType(
    instrument_id=instrument.id,
    bar_spec=BarSpecification(
        step=1,
        aggregation=BarAggregation.HOUR,
        price_type=PriceType.LAST,
    ),
    aggregation_source=AggregationSource.EXTERNAL,
)

# Process klines to bars
bars = process_klines_to_bars(klines, instrument, bar_type)
print(f"Processed {len(bars)} bars")

if bars:
    print(f"First bar: {bars[0]}")
    print(f"Last bar: {bars[-1]}")

## Setting Up Data Catalog

Store the processed data in a ParquetDataCatalog for efficient backtesting:

In [ ]:
# Set up data catalog
CATALOG_PATH = os.getcwd() + "/backpack_connector_catalog"

# Clear if it already exists, then create fresh
if os.path.exists(CATALOG_PATH):
    shutil.rmtree(CATALOG_PATH)
os.mkdir(CATALOG_PATH)

# Create a catalog instance
catalog = ParquetDataCatalog(CATALOG_PATH)

# Write instrument and bars to catalog
catalog.write_data([instrument])
catalog.write_data(bars)

print(f"Data catalog created at: {CATALOG_PATH}")
print(f"Instruments in catalog: {catalog.instruments()}")

## Configuring and Running Backtest

Configure the backtest with the properly loaded instrument and data:

In [ ]:
# Configure data
data_configs = [
    BacktestDataConfig(
        catalog_path=CATALOG_PATH,
        data_cls=Bar,
        instrument_id=instrument.id,
        bar_spec=bar_type.spec,
    )
]

# Configure venue with Backpack-specific settings
venues_configs = [
    BacktestVenueConfig(
        name="BACKPACK",
        oms_type="NETTING",
        account_type="CASH",
        base_currency=None,
        starting_balances=["10000 USDC", "0.1 BTC"],
    )
]

# Configure strategy
strategies = [
    ImportableStrategyConfig(
        strategy_path="nautilus_trader.examples.strategies.ema_cross:EMACross",
        config_path="nautilus_trader.examples.strategies.ema_cross:EMACrossConfig",
        config={
            "instrument_id": str(instrument.id),
            "bar_type": str(bar_type),
            "fast_ema_period": 10,
            "slow_ema_period": 20,
            "trade_size": Decimal("0.001"),  # Trade 0.001 BTC per signal
        },
    ),
]

# Create run configuration
config = BacktestRunConfig(
    engine=BacktestEngineConfig(
        strategies=strategies,
        logging=LoggingConfig(log_level="INFO"),
    ),
    data=data_configs,
    venues=venues_configs,
)

print("Backtest configuration ready")

In [ ]:
# Run backtest
node = BacktestNode(configs=[config])
result = node.run()

print(f"\nBacktest completed!")
if result:
    print(f"Run ID: {result[0].run_id}")

## Analyzing Results

In [ ]:
from nautilus_trader.backtest.engine import BacktestEngine

# Get engine for detailed reports
engine: BacktestEngine = node.get_engine(config.id)

# Generate reports
fills_report = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report = engine.trader.generate_account_report(BACKPACK_VENUE)

print("\n=== Order Fills Report ===")
if not fills_report.empty:
    print(fills_report.head(10))
else:
    print("No trades executed")

print("\n=== Positions Report ===")
if not positions_report.empty:
    print(positions_report.head(10))
else:
    print("No positions opened")

print("\n=== Account Report ===")
print(account_report)

## Summary

In this tutorial, we demonstrated how to use the **official Backpack Exchange connector** for backtesting:

### Key Components Used

1. **BackpackHttpClient**: Handles HTTP communication with the Backpack API
2. **BackpackHistoryHttpAPI**: Provides access to historical data endpoints
3. **CurrencyPair**: Proper instrument definition with auto-detected precision
4. **ParquetDataCatalog**: Efficient data storage for backtesting

### Advantages of Using the Connector

- **Production-ready**: Uses the same infrastructure as live trading
- **Type safety**: Proper instrument definitions with accurate precision
- **Integration**: Seamless integration with NautilusTrader ecosystem
- **Extensibility**: Easy to extend for live trading

### Comparison with Direct API Approach

| Aspect | Direct API (First Tutorial) | Connector (This Tutorial) |
|--------|---------------------------|-------------------------|
| Complexity | Simple, transparent | More setup required |
| Authentication | Not required | Dummy credentials needed |
| Error Handling | Basic | Built-in retries |
| Production Ready | No | Yes |
| Live Trading | Requires rewrite | Same code works |

### Next Steps

- Get real API credentials for accessing private data
- Explore more data types (trades, order book deltas)
- Implement custom strategies using the connector
- Transition to live trading with `BackpackLiveDataClient`